In [ ]:
#Member1: <Hana Khaled> 20240650
#Member2: <Jana Ahmed> 20240759

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [2]:
from sklearn.datasets import load_breast_cancer

In [3]:
data = load_breast_cancer()
X = data.data
y= data.target


In [4]:
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print("feature names:", data.feature_names)
print("classes: ", data.target_names)

Shape of X: (569, 30)
Shape of y: (569,)
feature names: ['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness' 'mean compactness' 'mean concavity'
 'mean concave points' 'mean symmetry' 'mean fractal dimension'
 'radius error' 'texture error' 'perimeter error' 'area error'
 'smoothness error' 'compactness error' 'concavity error'
 'concave points error' 'symmetry error' 'fractal dimension error'
 'worst radius' 'worst texture' 'worst perimeter' 'worst area'
 'worst smoothness' 'worst compactness' 'worst concavity'
 'worst concave points' 'worst symmetry' 'worst fractal dimension']
classes:  ['malignant' 'benign']


In [ ]:

def train_test_split(X, y, test_size=0.2, seed=42):

    np.random.seed(seed)

    classes = np.unique(y)

    train_indices = []
    test_indices = []

    # Calculate how many samples we need in the test set
    total_test = round(len(y) * test_size)

    # Get the number of test samples for each class
    class_indices = {}
    test_counts = {}

    for c in classes:
        indices = np.where(y == c)[0]
        np.random.shuffle(indices)

        class_indices[c] = indices

        # Initial allocation using floor
        test_counts[c] = int(len(indices) * test_size)

    # Distribute any remaining test samples
    remaining = total_test - sum(test_counts.values())

    fractions = {
        c: len(class_indices[c]) * test_size - test_counts[c]
        for c in classes
    }

    for c in sorted(classes, key=lambda c: fractions[c], reverse=True):
        if remaining == 0:
            break
        test_counts[c] += 1
        remaining -= 1

    for c in classes:
        indices = class_indices[c]
        n_test = test_counts[c]

        test_indices.extend(indices[:n_test])
        train_indices.extend(indices[n_test:])

    np.random.shuffle(train_indices)
    np.random.shuffle(test_indices)

    X_train = X[train_indices]
    X_test = X[test_indices]

    y_train = y[train_indices]
    y_test = y[test_indices]

    return X_train, X_test, y_train, y_test


In [36]:
X_train, X_test, y_train, y_test = train_test_split(X, y)

In [37]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (455, 30)
X_test: (114, 30)
y_train: (455,)
y_test: (114,)


In [61]:
print("Original distribution:")
print(np.unique(y, return_counts=True))

print("Training distribution:")
print(np.unique(y_train, return_counts=True))

print("Testing distribution:")
print(np.unique(y_test, return_counts=True))

Original distribution:
(array([0, 1]), array([212, 357]))
Training distribution:
(array([0, 1]), array([169, 286]))
Testing distribution:
(array([0, 1]), array([43, 71]))


In [40]:
#calculating the prior probabilities of each class
prior_0 = y_train[y_train == 0].size / y_train.size
prior_1 = y_train[y_train == 1].size / y_train.size

In [41]:
X_train_malignant = X_train[y_train==0]
X_train_benign = X_train[y_train==1]

In [42]:
print(X_train_malignant.shape)
print(X_train_benign.shape)

(169, 30)
(286, 30)


In [43]:
mean_malignant = np.mean(X_train_malignant, axis=0)
mean_benign = np.mean(X_train_benign, axis=0)

In [44]:
print(mean_malignant.shape)
print(mean_benign.shape)

(30,)
(30,)


In [47]:
variance_malignant = np.var(X_train_malignant, axis=0)
variance_benign = np.var(X_train_benign, axis=0)

In [49]:
#variance smoothing to avoid zero variance
var_smoothing = 1e-9
epsilon = var_smoothing * np.var(X_train, axis=0).max()
variance_malignant += epsilon
variance_benign += epsilon

print("Epsilon:", epsilon)
print("Malignant variances:", variance_malignant)
print("Benign variances:", variance_benign)

Epsilon: 0.00030920868184538094
Malignant variances: [1.05515221e+01 1.42094050e+01 4.95082616e+02 1.41324491e+05
 4.73623907e-04 3.05941124e-03 5.94564903e-03 1.54936418e-03
 9.86948981e-04 3.64192051e-04 1.31230670e-01 2.44934429e-01
 7.19849121e+00 4.18922801e+03 3.18653745e-04 6.66067158e-04
 7.84492476e-04 3.40447755e-04 3.99018452e-04 3.13568130e-04
 1.79745800e+01 3.01706982e+01 8.67917216e+02 3.49756250e+05
 8.19321761e-04 2.83234715e-02 3.18996265e-02 2.56688092e-03
 5.64869780e-03 7.38197261e-04]
Benign variances: [3.24441339e+00 1.73330315e+01 1.41832976e+02 1.81441094e+04
 5.01262307e-04 1.39166757e-03 2.16642144e-03 5.62922591e-04
 9.29067115e-04 3.55268563e-04 1.26255721e-02 3.66224552e-01
 5.76530303e-01 7.12981841e+01 3.18592965e-04 5.44639796e-04
 1.50993179e-03 3.43193959e-04 3.55972079e-04 3.16168709e-04
 3.97056640e+00 3.10083680e+01 1.80332587e+02 2.67163383e+04
 7.23831541e-04 8.51680148e-03 2.03630999e-02 1.58520907e-03
 2.04734648e-03 5.02459858e-04]


In [ ]:
#Gaussian Naive Bayes Classifier

class GaussianNaiveBayes:
    def __init__(self,var_smoothing=1e-9):
        self.var_smoothing = var_smoothing
        
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        
        self.class_prior_ = []
        for c in self.classes_:
            count = np.sum(y == c)
            prior = count / len(y)
            self.class_prior_.append(prior)
        
        self.class_prior_ = np.array(self.class_prior_)

        #mean and variance for each class
        self.theta_ = []
        self.var_ = []

        for c in self.classes_:
            X_class = X[y == c]
            class_mean = np.mean(X_class, axis=0)
            self.theta_.append(class_mean)

            class_var = np.var(X_class, axis=0)
            self.var_.append(class_var)

        self.theta_ = np.array(self.theta_)
        self.var_ = np.array(self.var_)

        self.epsilon_ = self.var_smoothing * np.max(np.var(X, axis=0))

        self.var_ = self.var_ + self.epsilon_

        return self

    def predict(self, X):
        scores = []

        for i in range(len(self.classes_)):
            mean = self.theta_[i]
            variance = self.var_[i]
            log_prior = np.log(self.class_prior_[i])
            log_likelihood = -0.5 * np.sum(np.log(2 * np.pi * variance)+ ((X - mean) ** 2 / variance),axis=1)
            class_score = log_prior + log_likelihood
            scores.append(class_score)

        scores = np.array(scores).T
        predictions = self.classes_[np.argmax(scores, axis=1)]

        return predictions

    def predict_proba(self, X):
        scores = []
        for i in range(len(self.classes_)):
            mean = self.theta_[i]
            variance = self.var_[i]
            log_prior = np.log(self.class_prior_[i])
            log_likelihood = -0.5 * np.sum(np.log(2 * np.pi * variance) + ((X - mean) ** 2 / variance), axis=1)
            scores.append(log_prior + log_likelihood)

        scores = np.array(scores).T                      
        m = np.max(scores, axis=1, keepdims=True)         
        exp_scores = np.exp(scores - m)
        return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

In [62]:
#sanity check

#small dataset created 
X_small = np.array([[1],[2],[3],[6],[7],[8]])
y_small = np.array([0, 0, 0, 1, 1, 1])
# 2. Manual calculations using a calculator
# Class 0 [1, 2, 3]
# mean_0 = (1 + 2 + 3) / 3 = 2
# var_0 = ((1-2)^2 + (2-2)^2 + (3-2)^2) / 3 = (1 + 0 + 1) / 3 = 0.666667
# Class 1 [6, 7, 8]
# mean_1 = (6 + 7 + 8) / 3 = 7
# var_1 = ((6-7)^2 + (7-7)^2 + (8-7)^2) / 3 = (1 + 0 + 1) / 3 = 0.666667
# prior_0 = 3 / 6 = 0.5
# prior_1 = 3 / 6 = 0.5

#Fit our Gaussian Naive Bayes model
small_model = GaussianNaiveBayes(var_smoothing=0)
small_model.fit(X_small, y_small)
print("Classes:", small_model.classes_)
print("Means:\n", small_model.mean_)
print("Variances:\n", small_model.var_)
print("Priors:", small_model.class_prior_)

# P(x|c) = 1 / sqrt(2*pi*var_c) *exp(-(x - mean_c)^2 / (2*var_c))
#class 0:
# P(2|0) = 1 / sqrt(2*pi*(2/3)) * exp(0) = 0.4886025
#class 1:
# P(2|1) = 1 / sqrt(2*pi*(2/3)) * exp(-(2-7)^2 / (2*(2/3))) = app. 3.515e-9
# Bayes' theorem:
# P(0|2) = P(2|0)*P(0) / (P(2|0)*P(0) + P(2|1)*P(1)) =  app. 0.999999993
#the predicted class should be 0.

#Test model 
new_sample = np.array([[2]])
prediction = small_model.predict(new_sample)
print("\nNew sample:", new_sample)
print("Predicted class:", prediction)
# Expected prediction: [0]


Classes: [0 1]
Means:
 [[2.]
 [7.]]
Variances:
 [[0.66666667]
 [0.66666667]]
Priors: [0.5 0.5]

New sample: [[2]]
Predicted class: [0]
